# Task-8 Week-3.

**Dataset :-** 'Admission_Prediction.csv'

**Objective :-Implement survival analysis techniques to model and predict student retention and dropout risks based on demographic and academic features from admission data.

In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.utils import concordance_index

In [21]:
df = pd.read_csv("Admission_Prediction.csv")

print("Dataset shape:", df.shape)
display(df.head())
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (1000, 14)


,Serial No.,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research,Chance of Admit,Year,Application_Date,University_Name,region,Admission Status
0,104,317,104,2,4.5,4.0,8.47,0,0.57,2022,2022-01-03,Indian Institute of Technology Bombay,West,Rejected
1,170,311,99,2,2.5,3.0,7.98,0,0.65,2022,2022-01-09,Indian Institute of Technology Delhi,North,Rejected
2,138,316,100,2,1.5,3.0,8.16,1,0.71,2022,2022-01-12,Indian Institute of Technology Delhi,North,Rejected
3,147,315,105,3,2.0,2.5,8.48,0,0.75,2022,2022-01-13,Indian Institute of Technology Delhi,North,Accepted
4,117,299,102,3,4.0,3.5,8.62,0,0.56,2022,2022-01-14,Indian Institute of Technology Delhi,North,Rejected



Columns:
['Serial No.', 'GRE Score', 'TOEFL Score', 'University Rating', 'SOP', 'LOR', 'CGPA', 'Research', 'Chance of Admit', 'Year', 'Application_Date', 'University_Name', 'region', 'Admission Status']


In [22]:
# Inspect schema, missing values, and candidate features

audit = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_values": df.isna().sum(),
    "unique_values": df.nunique(dropna=True)
})
display(audit)
display(df.describe(include="all").T)

,dtype,missing_values,unique_values
Serial No.,int64,0,1000
GRE Score,int64,0,51
TOEFL Score,int64,0,29
University Rating,int64,0,5
SOP,float64,0,27
LOR,float64,0,26
CGPA,float64,0,249
Research,int64,0,2
Chance of Admit,float64,0,65
Year,int64,0,5


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Serial No.,1000.0,NaN,NaN,NaN,500.5,288.819436,1.0,250.75,500.5,750.25,1000.0
GRE Score,1000.0,NaN,NaN,NaN,315.965,13.415653,290.0,305.0,316.0,326.25,340.0
TOEFL Score,1000.0,NaN,NaN,NaN,106.564,7.634277,92.0,101.0,107.0,113.0,120.0
University Rating,1000.0,NaN,NaN,NaN,3.015,1.317768,1.0,2.0,3.0,4.0,5.0
SOP,1000.0,NaN,NaN,NaN,3.3935,1.027279,0.9,2.5,3.5,4.0,5.1
LOR,1000.0,NaN,NaN,NaN,3.4067,0.891864,0.9,3.0,3.5,4.0,5.1
CGPA,1000.0,NaN,NaN,NaN,8.58979,0.61217,6.77,8.18,8.58,9.07,9.95
Research,1000.0,NaN,NaN,NaN,0.503,0.500241,0.0,0.0,1.0,1.0,1.0
Chance of Admit,1000.0,NaN,NaN,NaN,0.71777,0.14591,0.32,0.6375,0.72,0.82,0.97
Year,1000.0,NaN,NaN,NaN,2024.0,1.414921,2022.0,2023.0,2024.0,2025.0,2026.0


In [23]:
# Confirm whether survival outcomes exist

DURATION_COL = "Duration_Months"
EVENT_COL = "Dropout_Event"

required_outcomes = [DURATION_COL, EVENT_COL]
missing_outcomes = [c for c in required_outcomes if c not in df.columns]

if missing_outcomes:
    print("SURVIVAL MODEL NOT FIT: required outcome columns are missing:", missing_outcomes)
    print("Current columns:", df.columns.tolist())
    print("\nNext step: join a verified student-level retention dataset using a reliable student ID.")
    print("Do not derive dropout from Admission Status or Application_Date.")
else:
    print("Outcome columns found. Validate their definitions and values before modeling.")
    display(df[[DURATION_COL, EVENT_COL]].describe(include="all"))

SURVIVAL MODEL NOT FIT: required outcome columns are missing: ['Duration_Months', 'Dropout_Event']
Current columns: ['Serial No.', 'GRE Score', 'TOEFL Score', 'University Rating', 'SOP', 'LOR', 'CGPA', 'Research', 'Chance of Admit', 'Year', 'Application_Date', 'University_Name', 'region', 'Admission Status']

Next step: join a verified student-level retention dataset using a reliable student ID.
Do not derive dropout from Admission Status or Application_Date.


In [24]:
# Data validation

if not missing_outcomes:
    df[DURATION_COL] = pd.to_numeric(df[DURATION_COL], errors="coerce")
    df[EVENT_COL] = pd.to_numeric(df[EVENT_COL], errors="coerce")

    print("Missing duration:", df[DURATION_COL].isna().sum())
    print("Missing event:", df[EVENT_COL].isna().sum())
    print("Event values:", sorted(df[EVENT_COL].dropna().unique().tolist()))

    assert df[DURATION_COL].notna().all(), "Duration contains missing/non-numeric values."
    assert (df[DURATION_COL] > 0).all(), "Duration must be positive."
    assert set(df[EVENT_COL].unique()).issubset({0, 1}), "Event must be coded 0/1."
    assert df[EVENT_COL].notna().all(), "Event indicator contains missing values."
    print("Basic outcome validation passed.")

In [25]:
# Select predictors and avoid leakage

candidate_features = [
    "GRE Score", "TOEFL Score", "University Rating", "SOP", "LOR",
    "CGPA", "Research", "region", "Year"
]
FEATURES = [c for c in candidate_features if c in df.columns]
print("Candidate predictors found:", FEATURES)
print("Candidate predictors missing:", sorted(set(candidate_features) - set(FEATURES)))

Candidate predictors found: ['GRE Score', 'TOEFL Score', 'University Rating', 'SOP', 'LOR', 'CGPA', 'Research', 'region', 'Year']
Candidate predictors missing: []


In [26]:
# Kaplan–Meier retention curve

if not missing_outcomes:
    kmf = KaplanMeierFitter()
    kmf.fit(
        durations=df[DURATION_COL],
        event_observed=df[EVENT_COL],
        label="Probability of not yet dropping out"
    )
    ax = kmf.plot_survival_function(ci_show=True)
    ax.set_title("Kaplan–Meier curve: dropout-free survival")
    ax.set_xlabel("Months since enrollment")
    ax.set_ylabel("Probability of remaining enrolled without dropout")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped: add and validate Duration_Months and Dropout_Event first.")

Skipped: add and validate Duration_Months and Dropout_Event first.


In [27]:
# Compare survival curves by a group

GROUP_COL = "Research"
if not missing_outcomes and GROUP_COL in df.columns:
    kmf = KaplanMeierFitter()
    fig, ax = plt.subplots(figsize=(8, 5))
    for group_value, group_df in df.dropna(subset=[GROUP_COL]).groupby(GROUP_COL):
        kmf.fit(group_df[DURATION_COL], group_df[EVENT_COL],
                label=f"{GROUP_COL}={group_value}")
        kmf.plot_survival_function(ax=ax, ci_show=False)
    ax.set_title(f"Dropout-free survival by {GROUP_COL}")
    ax.set_xlabel("Months since enrollment")
    ax.set_ylabel("Survival probability")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped: outcomes and grouping variable are required.")

Skipped: outcomes and grouping variable are required.


In [28]:
# Prepare model data and fit a Cox proportional-hazards model

if not missing_outcomes:
    model_df = df[FEATURES + [DURATION_COL, EVENT_COL]].copy()
    model_df = model_df.dropna(subset=[DURATION_COL, EVENT_COL])

    # Impute numeric/categorical predictors and one-hot encode categoricals for CoxPHFitter.
    numeric_features = model_df[FEATURES].select_dtypes(include=np.number).columns.tolist()
    categorical_features = [c for c in FEATURES if c not in numeric_features]

    for c in numeric_features:
        model_df[c] = pd.to_numeric(model_df[c], errors="coerce")
        model_df[c] = model_df[c].fillna(model_df[c].median())
    for c in categorical_features:
        model_df[c] = model_df[c].astype("string").fillna("Unknown")

    model_df = pd.get_dummies(model_df, columns=categorical_features, drop_first=True, dtype=float)
    model_df = model_df.replace([np.inf, -np.inf], np.nan).dropna()

    # lifelines expects the duration and event columns in the dataframe.
    cph = CoxPHFitter(penalizer=0.1)
    cph.fit(model_df, duration_col=DURATION_COL, event_col=EVENT_COL)
    display(cph.summary.sort_values("p", ascending=True))
    cph.plot()
    plt.title("Cox model coefficients (log hazard ratios)")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped: no valid survival outcomes yet.")

Skipped: no valid survival outcomes yet.


In [29]:
# Check proportional-hazards assumptions

if not missing_outcomes:
    cph.check_assumptions(model_df, p_value_threshold=0.05, show_plots=True)
else:
    print("Skipped: fit the Cox model after verified outcome columns are available.")

Skipped: fit the Cox model after verified outcome columns are available.


In [30]:
# Create risk ranking at a chosen follow-up horizon

HORIZON_MONTHS = 12

if not missing_outcomes:
    # cph expects exactly the encoded covariates used during fitting.
    covariate_cols = [c for c in model_df.columns if c not in [DURATION_COL, EVENT_COL]]
    X = model_df[covariate_cols]
    survival_at_horizon = cph.predict_survival_function(X, times=[HORIZON_MONTHS]).T.iloc[:, 0]
    risk_df = pd.DataFrame({
        "Estimated_dropout_risk_by_horizon": 1 - survival_at_horizon,
        "Observed_duration_months": model_df[DURATION_COL],
        "Observed_dropout_event": model_df[EVENT_COL]
    }, index=model_df.index).sort_values("Estimated_dropout_risk_by_horizon", ascending=False)
    display(risk_df.head(20))
else:
    print("Skipped: risk scores cannot be calculated without real outcomes.")

Skipped: risk scores cannot be calculated without real outcomes.
